# Bellman Expectation and Bellman Optimality for MDPs

This notebook provides a compact, course-ready implementation of:

- Bellman expectation updates for policy evaluation
- Bellman optimality updates for value iteration
- A small illustrative Markov Decision Process (MDP)

It is designed to be used in Google Colab or any standard Python environment.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

## 1. Define a small MDP

We use three states:

- `S0` = start state
- `S1` = intermediate state
- `S2` = terminal state

And two actions:

- `A0`
- `A1`

The transition model is represented by `P[s, a, s']`, and rewards by `R[s, a, s']`.


In [ ]:
states = ["S0", "S1", "S2"]
actions = ["A0", "A1"]

nS = len(states)
nA = len(actions)
gamma = 0.9

# Transition probabilities P[s, a, s']
P = np.zeros((nS, nA, nS))

# Reward function R[s, a, s']
R = np.zeros((nS, nA, nS))

# State S0
P[0, 0, 1] = 1.0  # S0 --A0--> S1
R[0, 0, 1] = 5.0

P[0, 1, 2] = 1.0  # S0 --A1--> S2
R[0, 1, 2] = 2.0

# State S1
P[1, 0, 2] = 1.0  # S1 --A0--> S2
R[1, 0, 2] = 4.0

P[1, 1, 0] = 1.0  # S1 --A1--> S0
R[1, 1, 0] = 1.0

# Terminal state S2 (self-loop)
P[2, :, 2] = 1.0
R[2, :, 2] = 0.0

P, R

## 2. Fixed policy for Bellman expectation updates

We evaluate a policy `pi[a|s]` using iterative Bellman expectation backups.

For this example:

- In `S0`, the policy chooses `A0` with probability 1
- In `S1`, the policy chooses `A1` with probability 1
- In `S2`, any action is equivalent because it is terminal


In [ ]:
# Policy matrix pi[s, a]
pi = np.array([
    [1.0, 0.0],  # S0
    [0.0, 1.0],  # S1
    [0.5, 0.5],  # S2
])

def bellman_expectation_backup(V, pi, P, R, gamma):
    new_V = np.zeros_like(V)
    for s in range(nS):
        total = 0.0
        for a in range(nA):
            for sp in range(nS):
                total += pi[s, a] * P[s, a, sp] * (R[s, a, sp] + gamma * V[sp])
        new_V[s] = total
    return new_V

def policy_evaluation(pi, P, R, gamma, theta=1e-8, max_iter=10_000):
    V = np.zeros(nS)
    for _ in range(max_iter):
        new_V = bellman_expectation_backup(V, pi, P, R, gamma)
        if np.max(np.abs(new_V - V)) < theta:
            return new_V
        V = new_V
    return V

V_pi = policy_evaluation(pi, P, R, gamma)
V_pi

## 3. Bellman optimality and value iteration

Value iteration repeatedly applies the Bellman optimality backup:


`V(s) = max_a sum_{s'} P(s'|s,a) [R(s,a,s') + gamma * V(s')]`


In [ ]:
def bellman_optimality_backup(V, P, R, gamma):
    new_V = np.zeros_like(V)
    for s in range(nS):
        action_values = []
        for a in range(nA):
            q = 0.0
            for sp in range(nS):
                q += P[s, a, sp] * (R[s, a, sp] + gamma * V[sp])
            action_values.append(q)
        new_V[s] = np.max(action_values)
    return new_V

def value_iteration(P, R, gamma, theta=1e-8, max_iter=10_000):
    V = np.zeros(nS)
    for _ in range(max_iter):
        new_V = bellman_optimality_backup(V, P, R, gamma)
        if np.max(np.abs(new_V - V)) < theta:
            return new_V
        V = new_V
    return V

V_star = value_iteration(P, R, gamma)
V_star

In [ ]:
def greedy_policy_from_values(V, P, R, gamma):
    policy = np.zeros((nS, nA))
    for s in range(nS):
        q_values = []
        for a in range(nA):
            q = 0.0
            for sp in range(nS):
                q += P[s, a, sp] * (R[s, a, sp] + gamma * V[sp])
            q_values.append(q)
        best_a = int(np.argmax(q_values))
        policy[s, best_a] = 1.0
    return policy

pi_star = greedy_policy_from_values(V_star, P, R, gamma)
pi_star

## 4. Interpreting the results

The arrays above show:

- the value function under the fixed policy `V_pi`
- the optimal value function `V_star`
- the greedy policy induced by `V_star`

You can replace the toy MDP with your course-specific examples or assignments.

In [ ]:
for i, s in enumerate(states):
    print(f"{s}: V_pi = {V_pi[i]:.4f}, V_star = {V_star[i]:.4f}, pi_star = {pi_star[i]}")

## 5. Optional extension

To extend this notebook for class use, you can add:

- stochastic transitions
- custom reward functions
- policy iteration
- comparison with Monte Carlo or temporal-difference methods
